# 02 — RoPE Geometry

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoharuni/forge-position/blob/main/notebooks/02_rope_geometry.ipynb)

Study rotary position embeddings as 2D rotations and verify the relative-position dot-product property.

**Runtime:** <5 minutes on free Colab T4

---

## 1. Install & Clone

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!git clone https://github.com/marcoharuni/forge-position.git
%cd forge-position
!/root/.local/bin/uv sync

## 2. Imports

In [ ]:
import sys
sys.path.insert(0, 'src')

import torch
from IPython.display import Image, display

from forge_position import (
    apply_rope,
    build_rope_cache,
    plot_add_vs_rotate,
    plot_attention_order_blindness,
    plot_rope_multi_frequency_clock,
    plot_rope_position_angles,
    plot_rope_relative_distance,
    plot_rope_rotation_circle,
)

print(f'PyTorch: {torch.__version__}')

## 3. Why position is needed

Without a positional signal, attention has no built-in way to distinguish reordered tokens.

In [ ]:
path = plot_attention_order_blindness()
display(Image(filename=str(path)))

## 4. Add vs rotate

Absolute embeddings add a position vector. RoPE rotates query and key pairs.

In [ ]:
path = plot_add_vs_rotate()
display(Image(filename=str(path)))

## 5. Position is an angle

In [ ]:
path = plot_rope_position_angles()
display(Image(filename=str(path)))

## 6. Rotation circle

In [ ]:
path = plot_rope_rotation_circle()
display(Image(filename=str(path)))

## 7. Relative distance geometry

In [ ]:
path = plot_rope_relative_distance()
display(Image(filename=str(path)))

## 8. Many frequency bands

RoPE uses many rotating 2D pairs, like clock hands moving at different speeds.

In [ ]:
path = plot_rope_multi_frequency_clock()
display(Image(filename=str(path)))

## 9. Relative-position sanity check

If every Q/K position is shifted by the same offset, RoPE attention scores should stay nearly unchanged.

In [ ]:
torch.manual_seed(0)
q = torch.randn(1, 2, 6, 8)
k = torch.randn(1, 2, 6, 8)
cos, sin = build_rope_cache(32, 8)

base_pos = torch.arange(6).unsqueeze(0)
shifted_pos = base_pos + 11

q1, k1 = apply_rope(q, k, cos, sin, position_ids=base_pos)
q2, k2 = apply_rope(q, k, cos, sin, position_ids=shifted_pos)

scores1 = q1 @ k1.transpose(-2, -1)
scores2 = q2 @ k2.transpose(-2, -1)
print((scores1 - scores2).abs().max())

## 10. Partial RoPE

In [ ]:
q_partial, k_partial = apply_rope(q, k, cos[:, :2], sin[:, :2], position_ids=base_pos, rotary_dim=4)
print('Partial RoPE output: batch=1, heads=2, tokens=6, head_dim=8')
print('Unrotated tail preserved:', bool(torch.allclose(q_partial[..., 4:], q[..., 4:])))

---

**Next →** [03_alibi_bias_visualization.ipynb](03_alibi_bias_visualization.ipynb)